## **Predicción de precios de vivienda en venta en Madrid**
### **Proyecto final de Machine Learning** -- The Bridge
#### **Alumnos:** Ana Manzanares | Ramiro Caruso

##### **Objetivo**
Estimar el precio de venta de una vivienda en Madrid. Se trata de un problema de regresión con target, es decir, supervisado.  
Este modelo permitirá a un comprador o vendedor disponer de una referencia objetiva del precio del inmueble.

##### **Datos**
Dataset de 11826 filas (anuncios) de vivienda en Madrid, obtenidos mediante scrapping de Idealista con las siguientes variables:
- provincia: Provincia del inmueble.
- zona: Distrito al que pertenece el inmueble.
- titulo: Título del anuncio.
- PrecioActual: Precio de venta actual. En euros. TARGET.
- PrecioAnterior: Precio previo en euros.
- metros: Superficie en metros cuadrados.
- habitaciones: Número de habitaciones.
- ascensor: Indica si el inmueble tiene ascensor.
- localizacion: Indica la orientación de la vivienda. (exterior o interior)
- planta: Planta del inmueble.
- baños: Número de baños.
- tags: Texto en etiquetas separadas por comas. Etiquetas del anuncio.
- descripcion: Descripción del anuncio.
- Enlace: URL del anuncio. Funciona como un ID.

##### **Estructura del proyecto**
1. Preprocesado y limpieza
2. Split
3. EDA
4. Modelado
5. Optimización
6. Evaluación final

Se indicarán las limitaciones del dataset en la evaluación final.


## 0. Dataset

### Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import sys
sys.path.append("src/utils")
from funciones import limpiar_ausentes

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import joblib

### El Dataset

In [2]:
df = pd.read_csv("src/data_sample/Datos.csv")
df

,provincia,zona,titulo,PrecioActual,PrecioAnterior,metros,habitaciones,ascensor,localizacion,planta,baños,tags,descripcion,Enlace
0,madrid,ciudad-lineal,"Piso en calle de San Marcelo, 22, Ventas, Madrid",355000,0,69,2.0,S,EXTERIOR,5ª,1,"VIVIENDA,LUMINOSO,VISTAS,REFORMADA,PORTERO,EXT...",Particular Vende vivienda totalmente reformada...,https://www.idealista.com/inmueble/106956987/
1,madrid,carabanchel,"Piso en calle Cabo Nicolás Mur, San Isidro, Ma...",149000,159000,91,3.0,N,EXTERIOR,1ª,0,"PISO,EXCLUSIVA,INMOBILIARIA,OPORTUNIDAD",Inmobiliarias Encuentro vende en exclusiva la ...,https://www.idealista.com/inmueble/106906044/
2,madrid,centro,"Piso en calle de Rodas, Lavapiés-Embajadores, ...",195000,0,36,1.0,S,NaN,2ª,0,"EXCLUSIVA,ESTUDIO",ESTUDIO EN PLENO CENTRO DE MADRIDSarago Servic...,https://www.idealista.com/inmueble/107306175/
3,madrid,usera,"Piso en calle de Ferroviarios, Almendrales, Ma...",195000,0,58,1.0,S,INTERIOR,BAJO,0,"VIVIENDA,HOGAR,FUNCIONAL","Esta acogedora vivienda, ubicada en una planta...",https://www.idealista.com/inmueble/106325171/
4,madrid,tetuan,"Dúplex en Bellas Vistas, Madrid",715000,750000,140,3.0,S,EXTERIOR,2ª,0,"TERRAZA,EXCLUSIVA,MODERNO","Maravilloso ATICO de reciente construcción, co...",https://www.idealista.com/inmueble/106627265/
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11821,madrid,centro,"Piso en calle del Amparo, Lavapiés-Embajadores...",189000,0,36,2.0,N,INTERIOR,BAJO,0,"PISO,INMOBILIARIA,HOGAR,NUEVO",¡Descubre tu nuevo hogar en el corazón de Madr...,https://www.idealista.com/inmueble/107292488/
11822,madrid,centro,"Piso en calle Gran Vía, Chueca-Justicia, Madrid",2600000,0,245,2.0,S,EXTERIOR,2ª,0,"PISO,LUJO,EXCLUSIVO,EXTERIOR",Piso totalmente exterior ubicado entre el barr...,https://www.idealista.com/inmueble/103878128/
11823,madrid,tetuan,"Piso en calle del Capitán Blanco Argibay, Vald...",219000,225000,56,1.0,S,EXTERIOR,BAJO,0,"APARTAMENTO,METRO",¡IDEAL INVERSORES! Precioso apartamento en Val...,https://www.idealista.com/inmueble/107025446/
11824,madrid,carabanchel,"Piso en calle de Aceuchal, Vista Alegre, Madrid",165000,0,74,3.0,N,EXTERIOR,2ª,0,"OPORTUNIDAD,METRO","**IDEAL INVERSORES, NO DEJES PASAR ESTA OPORTU...",https://www.idealista.com/inmueble/106946149/


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11826 entries, 0 to 11825
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   provincia       11826 non-null  str    
 1   zona            11826 non-null  str    
 2   titulo          11826 non-null  str    
 3   PrecioActual    11826 non-null  int64  
 4   PrecioAnterior  11826 non-null  int64  
 5   metros          11826 non-null  int64  
 6   habitaciones    11460 non-null  float64
 7   ascensor        11033 non-null  str    
 8   localizacion    10730 non-null  str    
 9   planta          10601 non-null  str    
 10  baños           11826 non-null  int64  
 11  tags            11664 non-null  str    
 12  descripcion     11761 non-null  str    
 13  Enlace          11826 non-null  str    
dtypes: float64(1), int64(4), str(9)
memory usage: 1.3 MB


In [4]:
df.describe()

,PrecioActual,PrecioAnterior,metros,habitaciones,baños
count,1.182600e+04,1.182600e+04,11826.000000,11460.000000,11826.000000
mean,1.030501e+06,7.359704e+04,153.790039,2.847731,0.394047
std,1.237718e+06,3.639753e+05,766.217750,1.432402,0.882134
min,1.200000e+04,0.000000e+00,11.000000,1.000000,0.000000
25%,2.890000e+05,0.000000e+00,68.000000,2.000000,0.000000
50%,6.200000e+05,0.000000e+00,103.000000,3.000000,0.000000
75%,1.329000e+06,0.000000e+00,160.000000,3.000000,0.000000
max,2.300000e+07,8.450000e+06,75000.000000,20.000000,7.000000


In [5]:
df.isnull().sum().sort_values(ascending=False)

planta            1225
localizacion      1096
ascensor           793
habitaciones       366
tags               162
descripcion         65
metros               0
PrecioAnterior       0
PrecioActual         0
titulo               0
zona                 0
provincia            0
baños                0
Enlace               0
dtype: int64

### Primer análisis

1. COLUMNAS:
- provincia: Provincia del inmueble. Siempre es Madrid (es constante).
- zona: Categórica. Distrito al que pertenece el inmueble.
- titulo: Texto. Título del anuncio.
- PrecioActual: Numérica. Precio de venta actual. En euros. TARGET.
- PrecioAnterior: Numérica. Precio previo en euros.
- metros: Numérica. Superficie en metros cuadrados.
- habitaciones: Numérica. Número de habitaciones.
- ascensor: Categórica binaria. Indica si el inmueble tiene ascensor.
- localizacion: Categórica binaria. Indica la orientación de la vivienda. (exterior o interior)
- planta: Texto. Planta del inmueble.
- baños: Numérica. Número de baños.
- tags: Texto en etiquetas separadas por comas. Etiquetas del anuncio.
- descripcion: Texto. Descripción del anuncio.
- Enlace: "Texto". URL del anuncio. Funciona como un ID.

2. LOS DATOS

- Dataset con 11.826 filas y 14 columnas
- Nulos en: planta, localizacion, ascensor, habitaciones, tags, descripcion.
- Los nulos de habitaciones, al ser NaN, pandas lo carga como un float en vez de un int.
- PrecioActual va desde 12.000 a 23.000.000 euros. Media muy desviada (1.030.501) respecto a la mediana (620.000). Asimetría fuerte.
- PrecioAnterior: Media de 73.597, pero los percentiles valen todos 0. Más del 75% de los datos son ceros.
- metros: Media de 154 respecto a la mediana de 103. Desviación estándar de 766. El máximo es 75.000, algo imposible para una vivienda. El mínimo son 11m2, pero ese es, lamentablemente, más razonable.
- habitaciones: de 1 a 20 y mediana 3. El máximo es "plausible".
- baños: mínimo 0, mediana 9 y percentil de 75 en 0 también. Una vivienda no puede tener 0 baños. Hay que investigar.  

## 1. Preprocesado y Limpieza

Partimos de un dataset de 11.826 filas. El objetivo es dejar los datos listos para el modelado tomando decisiones justificadas en base a evidencias estudiadas previamente en los Notebooks.

#### COLUMNA TÍTULO

El título de cada anuncio tiene una estructura constante que permite extraer variables:  
1. Tipo de inmueble
2. Barrio
Se trata de una transformación fila a fila, por tanto, no hay riesgo de data leakage. Se puede realizar antes del split.

In [ ]:
# Se puede sacar lo que va antes y después del "en" y obtener el tipo de inmueble
df["tipo_inmueble"] = df["titulo"].str.split(" en ").str[0].str.strip()

df["tipo_inmueble"].value_counts()

tipo_inmueble
Piso                           9551
Ático                           782
Dúplex                          435
Estudio                         339
Casa o chalet independiente     320
Chalet adosado                  213
Chalet pareado                  171
Chalet                           14
Casa rural                        1
Name: count, dtype: int64

In [ ]:
# Obtenemos Barrio quitando ", Madrid" del final, quedándonos con el último tramo
# Si lleva incluido "Piso en" o cualquier parecido, nos quedamos con lo posterior al último "en"
sin_madrid = df["titulo"].str.rsplit(",", n=1).str[0]
barrio = sin_madrid.str.rsplit(",", n=1).str[-1].str.strip()
df["barrio"] = barrio.str.rsplit(" en ", n=1).str[-1].str.strip()

df["barrio"].value_counts()

barrio
Goya                    612
Recoletos               467
Malasaña-Universidad    451
Castellana              415
Lavapiés-Embajadores    366
                       ... 
El Pardo                  5
Valdecarros               5
Pavones                   5
Aeropuerto                4
Horcajo                   2
Name: count, Length: 139, dtype: int64

### COLUMNA BAÑOS
El 78.7% de la columna vale 0. Una vivienda no puede tener 0 baños, así que es un valor "centinela".  
Se confirma comparando contra el precio: Si el 0 fuera real, esas viviendas serían las más baratas. En cambio, son todo lo contrario.

In [8]:
df.groupby("baños")["PrecioActual"].median()

baños
0     598000.0
1     330000.0
2     725000.0
3    1335000.0
4    2450000.0
5    2100000.0
6    2500000.0
7    2375500.0
Name: PrecioActual, dtype: float64

In [9]:
# Copia de baños con el 0 convertido en NaN. No es una imputación porque es un cálculo fila a fila
df["baños_limpio"] = df["baños"].replace(0, np.nan)

### COLUMNA PRECIOANTERIOR
El 90% de la columna vale 0. De las que tienen valor, el 100% son rebajas, lo cual no es verosímil.  
**El portal está generando un anuncio nuevo cada vez que una vivienda sube de precio, pero conserva el mismo anuncio cuando baja. Así, todo parecen rebajas y nunca registran subidas de precio en el mismo inmueble.**  
La columna no es informativa del precio anterior, sino de su rebaja. La podemos reducir a una binaria.

In [10]:
# De las que tienen valor, comprobamos que el 100% son bajadas de precio
con_valor = df[df["PrecioAnterior"] > 0]
bajada_precio = (con_valor["PrecioAnterior"] > con_valor["PrecioActual"]).sum()
print("Filas con PrecioAnterior:", len(con_valor))
print("Bajadas de precio:", bajada_precio)

Filas con PrecioAnterior: 1113
Bajadas de precio: 1113


In [11]:
# La columna solamente nos dice si hay o no rebaja. Operación fila a fila.
df["flag_rebaja"] = (df["PrecioAnterior"] > 0).astype(int)

### COLUMNAS: ASCENSOR, LOCALIZACIÓN Y PLANTA
Estas tres columnas tienen NaN que no son errores, sino que estructuralmente están "mal". Por ejemplo, en un chalet no existe "ascensor". Son características que no aplican.  
Se puede confirmar cruzando nulos con el tipo de inmueble. Casi todos se concentran en casas o chalets.  
Aplicaremos la función "limpiar_ausentes", que asigna NO_APLICA a chalets y casas, y DESCONOCIDO al resto.

In [ ]:
# Este es el patrón que se repite en las tres columnas. Estudiado en el notebook preprocessing.ipynb
df["ascensor_es_nulo"] = df["ascensor"].isna()
(df.groupby("tipo_inmueble")["ascensor_es_nulo"].mean() * 100).round(2)

tipo_inmueble
Casa o chalet independiente    100.00
Casa rural                     100.00
Chalet                         100.00
Chalet adosado                 100.00
Chalet pareado                 100.00
Dúplex                           0.00
Estudio                          5.60
Piso                             0.58
Ático                            0.00
Name: ascensor_es_nulo, dtype: float64

In [13]:
# Aplicamos la función a todas las columnas
chalet_casa = ["Casa o chalet independiente", "Chalet adosado", "Chalet pareado", "Chalet", "Casa rural"]
es_chalet_casa = df["tipo_inmueble"].isin(chalet_casa)

df = limpiar_ausentes(df, "ascensor", es_chalet_casa)
df = limpiar_ausentes(df, "localizacion", es_chalet_casa)
df = limpiar_ausentes(df, "planta", es_chalet_casa)

df = df.drop(columns="ascensor_es_nulo")

In [ ]:
# SÓTANO y -1 son la misma planta escrita de dos formas
# Unificamos y listo
df["planta_limpio"] = df["planta_limpio"].replace("SÓTANO", "-1")

df["planta_limpio"].value_counts(dropna=False)

planta_limpio
1ª             2108
2ª             1779
3ª             1570
BAJO           1509
4ª             1266
5ª              816
NO_APLICA       719
6ª              574
DESCONOCIDO     506
7ª              289
ENTREPLANTA     163
-1              147
8ª              124
9ª              106
10ª              47
11ª              35
12ª              22
13ª              16
14ª               8
15ª               7
17ª               6
16ª               2
-2                2
20ª               1
27ª               1
22ª               1
21ª               1
18ª               1
Name: count, dtype: int64

### COLUMNA HABITACIONES
Los NaN aquí no son todos tan estructurales como en las anteriores. La mayoría son estudios o lofts que, por definición, tienen 0 habitaciones. Les asignamos 0, que sigue sin ser una imputación, sino la definición del tipo y por tanto no estamos imputando ni creando data leakage.  
El resto de NaNs son reales, por lo que se dejan así para imputarse más tarde en el pipeline (aunque terminaremos usando CatBoost, que no usa Pipeline como tal). Todo eso es post-split.

In [16]:
# Los estudios sin dato son 0 habitaciones por definición (una sola estancia). Fila a fila.
df["habitaciones_limpio"] = df["habitaciones"]
es_estudio = df["tipo_inmueble"] == "Estudio"
df.loc[es_estudio & df["habitaciones"].isna(), "habitaciones_limpio"] = 0

In [17]:
# Los NaN que restan se quedan como NaN y se imputan en el modelo.
df.loc[df["habitaciones_limpio"].isna(), "tipo_inmueble"].value_counts()

tipo_inmueble
Ático                          15
Dúplex                         10
Casa o chalet independiente     1
Chalet pareado                  1
Name: count, dtype: int64

### COLUMNA METROS

Dos tipos de errores hay aquí.
1. Superficies imposibles (pisos de más de 10.000m2) que no se pueden imputar sin inventar los datos.
2. Precios por metro cuadrado muy bajos (menos de 500€ el m2), que al inspeccionarlo resultan proindivisos o duplicados.  
En ambos casos estamos hablando de pocas filas que suponen muy poco para el dataset en su totalidad, por tanto, se eliminan.

In [18]:
# Superficies que no cuadran
# Todo este estudio se realizó en el notebook previamente.
metros_superior = df[df["metros"] > 10000]
metros_superior[["titulo", "PrecioActual", "metros", "tipo_inmueble", "Enlace"]]

,titulo,PrecioActual,metros,tipo_inmueble,Enlace
6979,"Piso en calle de Emilio Ferrari, s/n, Pueblo N...",192000,75000,Piso,https://www.idealista.com/inmueble/107196960/
7997,"Chalet adosado en Palomas, Madrid",1495000,33175,Chalet adosado,https://www.idealista.com/inmueble/106105382/


In [19]:
# Ahora los €/m2 absurdamente bajos
# Buscamos algún precio que esté muy muy por debajo del mercado
# Si los precios de Madrid a día de hoy no bajan de 2000€ el m2, teniendo en cuenta que este dataset debe ser algo anterior (no indica fecha)
# Mirando "a ojo" deberíamos estimar que todo aquello que esté por debajo de 500 €/m2 tiene que estar mal
# Fuentes del precio: https://www.bankinter.com/blog/finanzas-personales/comprar-casa-barata-madrid
euros_m2 = df["PrecioActual"] / df["metros"]
sus = df[euros_m2 < 500]
sus[["titulo", "PrecioActual", "metros", "tipo_inmueble", "Enlace"]]

,titulo,PrecioActual,metros,tipo_inmueble,Enlace
2714,"Estudio en San Diego, Madrid",114500,811,Estudio,https://www.idealista.com/inmueble/107028561/
3683,"Piso en calle Marmolistas, Arcos, Madrid",12000,61,Piso,https://www.idealista.com/inmueble/107240309/
6669,"Piso en calle Marmolistas, Arcos, Madrid",12000,61,Piso,https://www.idealista.com/inmueble/106859553/
6979,"Piso en calle de Emilio Ferrari, s/n, Pueblo N...",192000,75000,Piso,https://www.idealista.com/inmueble/107196960/
7750,"Piso en Entrevías, Madrid",161000,479,Piso,https://www.idealista.com/inmueble/101241888/
7997,"Chalet adosado en Palomas, Madrid",1495000,33175,Chalet adosado,https://www.idealista.com/inmueble/106105382/


In [ ]:
# Esto es para las filas que son estudios mal medidos, proindivisos y duplicados. Van fuera.
df = df[euros_m2 >= 500].copy()

**Sobre los Lofts:** En el notebook descubrimos los Lofts cuando estudiábamos la columna metros. La tratamos ahí mismo.  
Aquí, sin embargo, la trataremos junto a todos los flags por unificar.  
Esta columna fue descubierta al enterarnos que algunos estudios tenían precios y tamaños disparatados por m2. En su descripción indicaban que eran LOFTS, por lo que encontramos una nueva variable que podíamos extraer de esa información.

### COLUMNA DESCRIPCIÓN
La descripción es texto libre, pero contiene información útil aun así. El primer hallazgo vino con el análisis de metros, como ya explicamos en la celda de arriba.  
Extraeremos de aquí también las situaciones jurídicas especiales que afectan al precio:
- Nuda propiedad
- Proindiviso
- Subasta
- Actualmente okupada
- Ya alquilada

In [21]:
# Pasamos la información de descripción en minusculas
descripcion_lower = df["descripcion"].fillna("").str.lower()

# Extraemos loft y lo convertimos en una flag
df["flag_loft"] = descripcion_lower.str.contains("loft").astype(int)

In [22]:
# El resto de situaciones jurídicas explicadas en las celdas de arriba
# Okupada usa "okupa|ocupad" porque en las descripciones aparece con las dos grafias (k y c).
df["flag_nuda_propiedad"] = descripcion_lower.str.contains("nuda propiedad").astype(int)
df["flag_proindiviso"] = descripcion_lower.str.contains("proindiviso").astype(int)
df["flag_subasta"] = descripcion_lower.str.contains("subasta").astype(int)
df["flag_okupada"] = descripcion_lower.str.contains("okupa|ocupad").astype(int)
df["flag_alquilada"] = descripcion_lower.str.contains("alquilad").astype(int)

flags = ["flag_loft", "flag_nuda_propiedad", "flag_proindiviso", "flag_subasta", "flag_okupada", "flag_alquilada"]
df[flags].sum()

flag_loft              147
flag_nuda_propiedad     80
flag_proindiviso         5
flag_subasta            26
flag_okupada           271
flag_alquilada         305
dtype: int64

### COLUMNA TAGS

La columna tags tiene etiquetas del portal separadas por comas (reformado, terraza, lujo...). Es un tipo de información que acompaña al precio.  
La binarizamos. Una columna por etiqueta que aparezca, al menos, el 1% de los anuncios (porque las raras no generalizan).  
Transformación fila a fila, por lo que no perdemos datos. Puede ir antes del split.

In [ ]:
# Extraemos las etiquetas y su frecuencia
todas_etiquetas = df["tags"].fillna("").str.split(",").explode().str.strip()
todas_etiquetas = todas_etiquetas[todas_etiquetas != ""]
frecuencia = todas_etiquetas.value_counts()

# Una flag por etiqueta presente en >=1% de los anuncios
etiquetas_utiles = frecuencia[frecuencia >= 100].index.tolist()

tags_texto = df["tags"].fillna("")
flags_tags = []
# Bucle sencillito y renombramos con "tag_" + la etiqueta que corresponda
for etiqueta in etiquetas_utiles:
    columna = "tag_" + etiqueta.lower()
    df[columna] = tags_texto.str.contains(etiqueta, regex=False).astype(int)
    flags_tags.append(columna)

df[flags_tags].sum().sort_values(ascending=False)

tag_piso                   6113
tag_vivienda               5719
tag_exterior               2516
tag_metro                  2236
tag_amplio                 1994
tag_terraza                1922
tag_reformado              1880
tag_oportunidad            1831
tag_exclusiva              1647
tag_luminoso               1435
tag_hogar                  1315
tag_espectacular           1231
tag_inmobiliaria           1204
tag_ático                  1095
tag_finca                  1094
tag_lujo                    986
tag_vistas                  937
tag_nuevo                   930
tag_exclusivo               874
tag_reformada               867
tag_equipada                859
tag_casa                    851
tag_parque                  800
tag_garaje                  792
tag_interior                792
tag_estrenar                725
tag_piscina                 721
tag_funcional               661
tag_hall                    659
tag_moderno                 649
tag_apartamento             638
tag_suit

### DUPLICADOS
Una misma vivienda, por lo que ya vimos antes de cómo funciona el portal, puede estar anunciada por varias agencias o pueden haber creado un anuncio nuevo para incrementarle el precio. Esto genera una URL distinta, por lo que al principio parece que no hay duplicados, pero sí los hay.  
Deshacemos los duplicados ANTES del split para que no se cuele uno en train y otro en test, por ejemplo. El modelo evaluaría algo que ya vio.  
Para encontrar estos duplicados, utilizamos 9 features y comparamos.

In [29]:
# Clave de 9 features. Si coinciden en todo esto, es el mismo inmueble anunciado varias veces
# No se incluyen tags porque son listas. Si variasen el orden, las tomaría como distintas
# No se incluye la descripcion porque es texto libre. A no ser que copien y peguen lo mismo, como escriban algo diferente, las tomaría como distintas.
# No se incluye la URL porque funciona como ID por cada publicación
clave = ["titulo", "PrecioActual", "metros", "habitaciones", "planta", "baños", "ascensor", "localizacion", "zona"]

sobrantes = df.duplicated(subset=clave, keep="first")
sobrantes.value_counts()

False    11184
True       636
Name: count, dtype: int64

In [30]:
# Conservamos la fila más completa de cada grupo
# Keep="first" se quedará con la que menos NaN tiene
df["n_ausentes"] = df.isna().sum(axis=1)
df = df.sort_values("n_ausentes")
df = df.drop_duplicates(subset=clave, keep="first").copy()
df = df.drop(columns="n_ausentes")

df

,provincia,zona,titulo,PrecioActual,PrecioAnterior,metros,habitaciones,ascensor,localizacion,planta,...,tag_balcones,tag_dúplex,tag_portero,tag_electrodomésticos,tag_goya,tag_solo_particulares,tag_ventanales,tag_parcela,tag_abstenerse_agencias,tag_seguridad
2232,madrid,moratalaz,"Piso en calle de la Hacienda de Pavones, Fonta...",250000,0,63,2.0,S,EXTERIOR,BAJO,...,0,0,0,0,0,0,0,0,0,0
2228,madrid,chamartin,"Piso en Bernabéu-Hispanoamérica, Madrid",4800000,0,463,4.0,S,EXTERIOR,2ª,...,0,1,0,0,0,0,0,0,0,0
2254,madrid,villa-de-vallecas,"Piso en avenida del Ensanche de Vallecas, Ensa...",420000,0,100,3.0,S,EXTERIOR,1ª,...,0,0,0,0,0,0,0,0,0,0
2274,madrid,barrio-de-salamanca,"Piso en Goya, Madrid",1459000,0,97,2.0,S,EXTERIOR,5ª,...,0,0,0,1,0,0,0,0,0,0
2266,madrid,barrio-de-salamanca,"Piso en Goya, Madrid",690000,0,60,2.0,S,INTERIOR,4ª,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8434,madrid,fuencarral,"Piso en calle de Joaquín Lorenzo, Peñagrande, ...",810000,0,131,3.0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
3623,madrid,chamartin,"Casa o chalet independiente en Nueva España, M...",3800000,0,391,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
2511,madrid,hortaleza,"Chalet adosado en calle Somontín, 51, Apóstol ...",390000,0,125,3.0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
8113,madrid,hortaleza,"Chalet pareado en Valdebebas - Valdefuentes, M...",775000,0,127,3.0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


### ESTADO DE LOS NULOS ANTES DEL SPLIT
Los nulos que quedan no son errores, sino que son nulos intencionales:
1. Tenemos las columnas originales sin tocar, que conservan sus NaN naturales. Su versión tratada (columna_limpio) está bien.
2. baños_limpio y habitaciones_limpio están esperando ser imputados post-split para no filtrar información del test.

In [ ]:
df.isnull().sum()

provincia                  0
zona                       0
titulo                     0
PrecioActual               0
PrecioAnterior             0
                          ..
tag_solo_particulares      0
tag_ventanales             0
tag_parcela                0
tag_abstenerse_agencias    0
tag_seguridad              0
Length: 82, dtype: int64

## SPLIT
Dividimos el Train y Test.
80/20, estratificado por zona para que ambos conjuntos tengan la misma composición por distrito y random_state en 42 como siempre.


In [ ]:
X = df.drop(columns="PrecioActual")
y = df["PrecioActual"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df["zona"])

train = X_train.copy()
train["PrecioActual"] = y_train

test = X_test.copy()
test["PrecioActual"] = y_test

In [ ]:
train.shape

In [ ]:
test.shape